In [23]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

In [24]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

# The aim of the assignment is to simulate the Ekert91 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.



In [25]:
# Generate Alice's bits and bases, and Bob's bases

def generate_bits_and_bases(n):
    """
    Bit values:  0 or 1
    Bases:       'S' → Simple basis:   |0⟩, |1⟩
                 'D' → diagonal basis: |+⟩, |−⟩
    """
    bits         = []
    bases        = []
    state_labels = []

    for _ in range(n):
        #Both bits and basis generation using quantum state for randomization.
        qc_bit = QuantumCircuit(1, 1)
        qc_bit.h(0)
        qc_bit.measure(0, 0)

        qc_basis = QuantumCircuit(1, 1)
        qc_basis.h(0)
        qc_basis.measure(0, 0)

        simulator = BasicSimulator()

        bit   = int(list(simulator.run(transpile(qc_bit,   simulator), shots=1).result().get_counts().keys())[0])
        basis = int(list(simulator.run(transpile(qc_basis, simulator), shots=1).result().get_counts().keys())[0])

        # Map (bit, basis) → state label
        state_map = {
            (0, 0): '0',   # S-basis, bit 0 → |0⟩
            (1, 0): '1',   # S-basis, bit 1 → |1⟩
            (0, 1): '+',   # D-basis, bit 0 → |+⟩
            (1, 1): '-',   # D-basis, bit 1 → |−⟩
        }

        bits.append(bit)
        bases.append('S' if basis == 0 else 'D')
        state_labels.append(state_map[(bit, basis)])

    return bits, bases, state_labels

# --- Run ---

n = 200
alice_bits, alice_bases, alice_states = generate_bits_and_bases(n)
_,          bob_bases,   _            = generate_bits_and_bases(n)

print(f"Alice bits:   {alice_bits}")
print(f"Alice bases:  {alice_bases}")
print(f"Alice states: {alice_states}")
print(f"Bob bases:    {bob_bases}")

Alice bits:   [1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1]
Alice bases:  ['S', 'S', 'S', 'D', 'S', 'D', 'D', 'D', 'D', 'D', 'S', 'D', 'S', 'D', 'S', 'D', 'S', 'D', 'D', 'D', 'S', 'S', 'S', 'D', 'S', 'D', 'S', 'D', 'D', 'S', 'S', 'D', 'D', 'S', 'S', 'D', 'D', 'D', 'D', 'D', 'D', 'D', 'S', 'S', 'D', 'D', 'D', 'D', 'D', 'S', 'S', 'D', 'S', 'S', 'S', 'D', 'D', 'S', 'S', 'S', 'S', 'D', 'S', 'D', 'S', 'D', 'D', 'D', 'S', 'D', 'D', 'S', 'D', 'D', 

In [26]:
#Sifting step where Bob measures using his own state and send to Alice to sift.

def bob_measure(alice_states, bob_bases):
    """
    Bob measures each incoming qubit using his own chosen basis.
    - If basis matches Alice's: result agrees with Alice's bit
    - If basis differs:         result is random (50/50)
    """
    bob_results = []

    for state, basis in zip(alice_states, bob_bases):
        qc = QuantumCircuit(1, 1)

        # Prepare the incoming qubit in Alice's state
        if state == '1':
            qc.x(0)           # |1⟩
        elif state == '+':
            qc.h(0)           # |+⟩
        elif state == '-':
            qc.x(0)
            qc.h(0)           # |−⟩
        # state == '0': do nothing, already |0⟩

        # Bob applies his basis choice before measuring
        if basis == 'D':
            qc.h(0)           # rotate to D-basis before measuring

        qc.measure(0, 0)

        simulator = BasicSimulator()
        result = int(list(simulator.run(transpile(qc, simulator), shots=1).result().get_counts().keys())[0])
        bob_results.append(result)

    return bob_results


# --- Sifting step ---
def sift_keys(alice_bits, alice_bases, bob_bases, bob_results):
    sifted_positions  = []
    alice_sifted_key  = []
    bob_sifted_key    = []

    for i in range(len(alice_bits)):
        if alice_bases[i] == bob_bases[i]:
            sifted_positions.append(i)
            alice_sifted_key.append(alice_bits[i])
            bob_sifted_key.append(bob_results[i])

    return sifted_positions, alice_sifted_key, bob_sifted_key


# --- Run ---
bob_results = bob_measure(alice_states, bob_bases)

sifted_positions, alice_sifted_key, bob_sifted_key = sift_keys(
    alice_bits, alice_bases, bob_bases, bob_results
)

sifted_key = alice_sifted_key

print(f"Alice sifted key: {alice_sifted_key}")
print(f"Bob sifted key:   {bob_sifted_key}")
print(f"Keys match:       {alice_sifted_key == bob_sifted_key}")

Alice sifted key: [1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1]
Bob sifted key:   [1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1]
Keys match:       True


In [27]:
#Encoding and decoding message using the key

def text_to_bits(text):
    """Convert a string to a list of bits."""
    bits = []
    for char in text:
        # Convert each character to 8-bit binary
        char_bits = format(ord(char), '08b')
        bits.extend([int(b) for b in char_bits])
    return bits

def bits_to_text(bits):
    """Convert a list of bits back to a string."""
    text = ''
    for i in range(0, len(bits), 8):
        byte = bits[i:i+8]
        text += chr(int(''.join(map(str, byte)), 2))
    return text

def encrypt(message, key):
    """XOR message bits with key bits (one-time pad)."""
    message_bits = text_to_bits(message)

    # Key must be at least as long as the message
    if len(key) < len(message_bits):
        raise ValueError(f"Key too short: need {len(message_bits)} bits, only have {len(key)}")

    # XOR each message bit with corresponding key bit
    ciphertext = [m ^ k for m, k in zip(message_bits, key)]
    return ciphertext, message_bits

def decrypt(ciphertext, key):
    """XOR ciphertext bits with key bits to recover message."""
    decrypted_bits = [c ^ k for c, k in zip(ciphertext, key)]
    return bits_to_text(decrypted_bits)


# --- Run  ---
message = "Hello"
print(f"Original message:   {message}")
print(f"Sifted key length:  {len(sifted_key)} bits")
print(f"Message bit length: {len(text_to_bits(message))} bits")
print()

# Alice encrypts and sends message
ciphertext, message_bits = encrypt(message, sifted_key)

# Bob decrypts message
decrypted = decrypt(ciphertext, sifted_key)

print(f"Message bits: {message_bits}")
print(f"Ciphertext:   {ciphertext}")
print(f"Decrypted:    {decrypted}")

Original message:   Hello
Sifted key length:  123 bits
Message bit length: 40 bits

Message bits: [0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1]
Ciphertext:   [1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0]
Decrypted:    Hello
